# Complete Working Solution for Kaggle

In [1]:
# ========== CELL 1: Install Dependencies ==========
print("📦 Installing required system packages...")

# Install zstd and other dependencies
!apt-get update -qq
!apt-get install -y zstd curl wget -qq

print("✅ Dependencies installed!")

📦 Installing required system packages...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../wget_1.21.2-2ubuntu1.3_amd64.deb ...
Unpacking wget (1.21.2-2ubuntu1.3) over (1.21.2-2ubuntu1.1) ...
Preparing to unpack .../libcurl4-openssl-dev_7.81.0-1ubuntu1.25_amd64.deb ...
Unpacking libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.25) over (7.81.0-1ubuntu1.24) ...
Preparing to unpack .../curl_7.81.0-1ubuntu1.25_amd64.deb ...
Unpacking curl (7.81.0-1ubuntu1.25) over (7.81.0-1ubuntu1.24) ...
Preparing to unpack .../libcurl4_7.81.0-1ubuntu1.25_amd64.deb ...
Unpacking libcurl4:amd64 (7.81.0-1ubuntu1.25) over (7.81.0-1ubuntu1.24) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3

In [2]:
# ========== CELL 2: Install Ollama ==========
print("📦 Installing Ollama...")

# Remove any existing ollama to avoid conflicts
!rm -rf /usr/local/lib/ollama
!rm -f /usr/local/bin/ollama

# Install with proper extraction
!curl -fsSL https://ollama.com/install.sh | sh

# Wait and verify
import time
time.sleep(3)

# Check if installed
!which ollama

print("✅ Ollama installation complete!")

📦 Installing Ollama...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                      0.5%################                            64.6%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama
✅ Ollama installation complete!


In [3]:
# ========== CELL 3: Start Ollama Server ==========
import subprocess
import time
import os

def start_ollama_server():
    """Starts the Ollama server with proper error handling."""
    try:
        # Check if ollama command exists
        result = subprocess.run(['which', 'ollama'], capture_output=True, text=True)
        if not result.stdout:
            print("❌ Ollama not installed. Trying manual installation...")
            return manual_ollama_install()
        
        # Start the server
        print("🚀 Starting Ollama server...")
        process = subprocess.Popen(
            ['ollama', 'serve'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        time.sleep(8)  # Give it time to initialize
        
        # Check if server is running
        try:
            check = subprocess.run(['curl', '-s', 'http://localhost:11434'], 
                                  capture_output=True, timeout=5)
            if check.returncode == 0:
                print("✅ Ollama server is running!")
                return True
        except:
            pass
        
        print("⚠️ Server might still be starting... continuing anyway")
        return True
    except Exception as e:
        print(f"❌ Error starting server: {e}")
        return False

def manual_ollama_install():
    """Manual installation fallback if script fails."""
    print("⚙️ Attempting manual installation...")
    
    # Download the binary directly
    !wget -q https://ollama.com/download/ollama-linux-amd64 -O /tmp/ollama
    !chmod +x /tmp/ollama
    !sudo mv /tmp/ollama /usr/local/bin/ollama
    
    # Verify
    result = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ Ollama installed manually!")
        return True
    else:
        print("❌ Manual installation failed")
        return False

# Start the server
start_ollama_server()

🚀 Starting Ollama server...
✅ Ollama server is running!


True

In [4]:
# ========== CELL 4: Test the Installation ==========
print("\n🧪 Testing Ollama...")

# Show version
!ollama --version

# List any existing models
!ollama list

print("\n✅ Setup complete!")


🧪 Testing Ollama...
]11;?\ollama version is 0.32.1
]11;?\NAME    ID    SIZE    MODIFIED 

✅ Setup complete!


In [5]:
# ========== CELL 5: Pull and Test a Model ==========
print("\n📥 Pulling a small model (llama3.2:1b - ~1.4GB)...")

# Pull the model using subprocess to see progress
result = subprocess.run(['ollama', 'pull', 'llama3.2:1b'], 
                       capture_output=True, text=True)

if result.returncode == 0:
    print("✅ Model downloaded successfully!")
    
    # Test the model
    print("\n🧪 Testing model with a simple question...")
    test_result = subprocess.run(
        ['ollama', 'run', 'llama3.2:1b', 'What is 2+2? Answer briefly.'],
        capture_output=True,
        text=True
    )
    print(f"Response: {test_result.stdout.strip()}")
else:
    print(f"⚠️ Failed to pull model: {result.stderr}")
    print("Try running this manually: !ollama pull llama3.2:1b")


📥 Pulling a small model (llama3.2:1b - ~1.4GB)...
✅ Model downloaded successfully!

🧪 Testing model with a simple question...
Response: 2 + 2 = 4.


# Alternative: Using the Python Ollama Library

In [6]:
!pip install ollama

In [7]:
# ========== ALTERNATIVE: Pure Python Approach ==========
import subprocess
import time
import ollama

# 1. Install dependencies
!apt-get update -qq && apt-get install -y zstd curl wget -qq

# 2. Install Ollama (with zstd)
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start server in background
import os
os.system('ollama serve &')
time.sleep(10)

# 4. Use Python library
print("📥 Pulling model via Python...")
client = ollama.Client(host='http://localhost:11434')

try:
    client.pull('llama3.2:1b')
    print("✅ Model pulled!")
    
    # Test
    response = client.chat(
        model='llama3.2:1b',
        messages=[{'role': 'user', 'content': 'What is 2+2? Answer briefly.'}]
    )
    print(f"Response: {response['message']['content']}")
    
except Exception as e:
    print(f"Error: {e}")
    print("\nTrying command-line fallback...")
    !ollama pull llama3.2:1b
    !ollama run llama3.2:1b "What is 2+2?"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%#########                                          45.4%##############################                               60.5%######################################           88.5%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Error: listen tcp 127.0.0.1:11434: bind: address already in use


📥 Pulling model via Python...
✅ Model pulled!
Response: 2 + 2 = 4.


# One-Liner Complete Setup

In [8]:
# ========== SIMPLEST WORKING SETUP ==========
# Install zstd and Ollama
!apt-get install -y zstd -qq && curl -fsSL https://ollama.com/install.sh | sh

# Start server
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)

# Test
!ollama pull llama3.2:1b
!ollama run llama3.2:1b "Say hello in 5 words"

print("✅ Working!")

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                     10.0%                                                       21.9%##########################                               60.5%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB       